# Frontal band-power trends across visits

**What this notebook does.** It takes the lab's existing processing pipeline
(`eeg_report.py`, written by your colleague) and pulls the *numbers* out of it, instead of
only the PDF reports. Then it trends frontal delta / theta / alpha / beta power across
visits, separately for each participant.

**What it does NOT do.** It does not change any signal processing. Filtering, epoching,
artifact rejection and the multitaper spectral estimate all still happen inside
`eeg_report.py`, exactly as they do for the reports your colleague generates. That matters:
your numbers and her reports can never disagree, because they come from the same code.

**How to run it.** Top to bottom, one cell at a time (Shift+Enter). Each step says what it
does and what you should see.

---
### The two files this notebook needs, in the same folder as the notebook
| file | what it is |
|---|---|
| `eeg_report.py` | your colleague's pipeline, **plus** a new `kind="export"` option that writes the numbers to CSV. Nothing else was changed. |
| `eeg_trends.py` | the new analysis helpers (loading, trend figures, statistics). Open it any time - every function is short and commented. |

## Step 1 - Tell the notebook where things are

`DATA_FOLDER` is the folder holding the raw recordings. The pipeline searches it
*recursively*, so subfolders are fine. It looks for files whose names contain
`PIN <number>`, `V<number>`, `PRE` or `POST`, and `Go No-Go`.

`OUT_FOLDER` is where results get written. It will be created if it does not exist.

In [ ]:
DATA_FOLDER = "eeg_data"          # <-- CHANGE THIS to your recordings folder
OUT_FOLDER  = "my_analysis"       # results land here

import os
os.makedirs(OUT_FOLDER, exist_ok=True)
print("data folder exists:", os.path.isdir(DATA_FOLDER))
print("writing results to:", os.path.abspath(OUT_FOLDER))

## Step 2 - Check the pipeline can see your files

Before processing anything, confirm the filenames are being read correctly. If a
participant or visit is missing here, it is a **filename** problem, not a data problem -
and it is much cheaper to find out now.

In [ ]:
import glob, re, os

files = sorted(
    f for pat in ("**/*Go No-Go*.csv", "**/*Go No-Go*.csv.zip")
    for f in glob.glob(os.path.join(DATA_FOLDER, pat), recursive=True)
    if re.search(r"PIN\s*\d+", os.path.basename(f), re.I)
)
print(f"{len(files)} recording file(s) found\n")

seen = {}
for f in files:
    up = os.path.basename(f).upper()
    pid = re.search(r"PIN\s*(\d+)", up); vis = re.search(r"\bV(\d+)", up)
    ph  = "POST" if "POST" in up else ("PRE" if "PRE" in up else "??")
    key = int(pid.group(1)) if pid else None
    seen.setdefault(key, set()).add((int(vis.group(1)) if vis else None, ph))

for pid in sorted(k for k in seen if k is not None):
    visits = sorted({v for v, _ in seen[pid] if v is not None})
    both   = sorted({v for v in visits if (v, "PRE") in seen[pid] and (v, "POST") in seen[pid]})
    print(f"  PIN {pid:03d}: {len(visits):2d} visits {visits}")
    print(f"            {len(both):2d} with BOTH pre & post -> {both}")

## Step 3 - Export the numbers (the one new thing)

This runs your colleague's `process()` on every recording and writes two CSV files.
It is the same computation that produces her reports - it just saves the numbers instead
of drawing them.

**Run it on one participant first.** Processing is the slow part (roughly a few seconds
per recording), so check the output looks sane before committing to the whole study.
Change `participant=1` to `participant="all"` for the full run.

### The two files it writes

**`band_power_session.csv`** - one row per participant x visit x phase x condition x region x band

| column | meaning |
|---|---|
| `participant`, `visit`, `phase` | who, which visit, `pre` or `post` |
| `week` | Control / Heat week 1-3, from the visit number |
| `condition` | `Go` or `NoGo` |
| `region` | `frontal` (F3,F4), `central` (C3,Cz,C4), `posterior` (P3,P4) |
| `band` | delta / theta / alpha / beta |
| `abs_power_uv2` | **the number you want** - band power in uV^2, averaged over clean epochs |
| `n_epochs` | how many clean epochs it was averaged over (quality) |
| `pct_rejected` | share of epochs thrown out as artifact (quality) |

**`band_power_epochs.csv`** - one row per *individual clean epoch* (frontal No-Go only).
You need this only for error bars and within-session tests; the trends use the first file.

In [ ]:
import eeg_report

eeg_report.generate_reports(
    kind="export",
    participant=1,            # <-- start with one; change to "all" for the full study
    base_dir=DATA_FOLDER,
    out_dir=OUT_FOLDER,
    ask=False,
    show_links=False,
)

## Step 4 - Load the numbers and see what you actually have

`inventory()` is the reality check. `visits_usable` is the number that limits every
statistic later: it counts visits where **both** PRE and POST survived with at least 20
clean epochs. A participant with 3 usable visits cannot support a trend, no matter how
good the figure looks.

In [ ]:
import pandas as pd
import eeg_trends as T

sessions, epochs = T.load(OUT_FOLDER)
print("session-level rows:", len(sessions))

inv = T.inventory(sessions)
inv

## Step 5 - Prepare the analysis table

`prepare()` does four things, and it is worth knowing each one:

1. keeps **frontal**, **No-Go** rows (change `region=` / `condition=` to look elsewhere);
2. drops any session with fewer than 20 clean epochs - the same threshold your
   colleague's reports use, so your numbers match her figures;
3. drops sessions where fewer than all four bands survived, so shares are comparable;
4. adds `rel_power_pct` - each band's **share of total 0.5-30 Hz power**, in percent.

**Why relative power matters.** Absolute power depends on electrode contact, hair, gel and
placement, which change from day to day. A visit-to-visit rise in absolute power can be a
better-seated electrode rather than a change in the brain. Relative power divides that out.
Look at both: a real effect usually shows up in both, an artifact usually shows up only in
absolute.

In [ ]:
trends = T.prepare(sessions, region="frontal", condition="NoGo")
print(trends.shape[0], "rows kept")
trends.head(8)

## Step 6 - The core figure: one participant, four bands, across visits

Dashed = PRE, solid = POST. The y-axis is a log scale, because band power is
multiplicative - a doubling and a halving should look the same size, and on a linear axis
they do not.

Run the absolute version and the relative version and compare them.

In [ ]:
FIG_DIR = os.path.join(OUT_FOLDER, "figures")

for pid in sorted(trends.participant.unique()):
    T.plot_participant_trends(trends, pid, value="abs_power_uv2",  save_dir=FIG_DIR)
    T.plot_participant_trends(trends, pid, value="rel_power_pct", save_dir=FIG_DIR)

print("saved to", FIG_DIR)

## Step 7 - The acute exercise effect, and whether it drifts

`acute()` pairs each visit's PRE with its POST and computes **log2(POST / PRE)**:

- `0` = no change
- `+1` = power doubled after exercise
- `-1` = power halved after exercise

The log is not decoration. A rise from 10 to 20 and a fall from 20 to 10 are the same size
change, but as raw differences they are `+10` and `-10` against different baselines. In
log2 they are exactly `+1` and `-1`, so averaging across visits is meaningful.

The figure asks the follow-up question: does that acute response get *stronger or weaker*
as the study goes on?

In [ ]:
acute = T.acute(trends)
print(acute.shape[0], "paired visit x band observations")

for pid in sorted(acute.participant.unique()):
    T.plot_acute_trends(acute, pid, save_dir=FIG_DIR)

acute.head(8)

## Step 8 - One band, every participant side by side

Small multiples rather than one crowded chart: nine lines in nine colours are not
distinguishable, and each participant sits at a different power level anyway. Each panel
keeps its own y-scale, so compare the **shape** of a trend across panels, not its height.

In [ ]:
for band in T.BANDS:
    T.plot_band_across_participants(trends, band, value="abs_power_uv2",  save_dir=FIG_DIR)
    T.plot_band_across_participants(trends, band, value="rel_power_pct", save_dir=FIG_DIR)

print("saved to", FIG_DIR)

## Step 9 - Statistics, per participant

Two questions, two non-parametric tests - non-parametric because band power is skewed and
you have few visits, so a t-test's normality assumption is not safe here.

| column | question it answers |
|---|---|
| `wilcoxon_p` | Is there an acute PRE->POST effect at all? (signed-rank on log2 ratio vs 0, across visits) |
| `spearman_rho`, `spearman_p` | Does that effect trend across the study? (rank correlation with visit number) |
| `*_holm` | the same p-values after Holm correction for testing every participant x band |

**Read `n_visits` before any p-value.** With fewer than 6 visits the Wilcoxon test cannot
produce a p below 0.03 even if every single visit moves the same way - the result is a
statement about your sample size, not about the brain.

In [ ]:
stats_table = T.trend_table(acute, min_visits=5)
stats_table.to_csv(os.path.join(OUT_FOLDER, "per_participant_stats.csv"), index=False)
stats_table

## Step 10 - Group-level tests

`friedman_across_visits` is the repeated-measures test across visits: blocks are
participants, treatments are visits. It needs a **complete** block - every participant
having every visit - so it automatically keeps the largest run of visits shared by enough
participants, and tells you what it kept. **Read `n_participants` and `visits` first.**

For PRE vs POST there are only two related conditions, where a Friedman test reduces to a
sign test - so `wilcoxon_pre_post_group` is the correct test there, not Friedman.

In [ ]:
import pandas as pd

fried = pd.DataFrame([T.friedman_across_visits(trends, b, phase=ph)
                      for b in T.BANDS for ph in ("pre", "post")])
fried.to_csv(os.path.join(OUT_FOLDER, "friedman_across_visits.csv"), index=False)
fried

In [ ]:
prepost = pd.DataFrame([T.wilcoxon_pre_post_group(trends, b) for b in T.BANDS])
prepost.to_csv(os.path.join(OUT_FOLDER, "group_pre_vs_post.csv"), index=False)
prepost

## Step 11 - Read this before you write anything up

These are properties of the pipeline you are using. None of them is a bug; all of them
change what you may claim.

**1. Delta is the weakest band here, which is awkward given it is on your list.**
Epochs are 1.0 s long (250 samples at 250 Hz, -200 to +800 ms). The multitaper setting
(`NW=2`, 3 tapers) smooths the spectrum by +/- 2 Hz, so the effective resolution is about
**4 Hz wide** - wider than the entire delta band (0.5-4 Hz). In practice the delta estimate
is only 7 frequency bins spanning **0.98-3.91 Hz**, and it is heavily blended with theta.
Treat frontal delta as the least trustworthy of your four traces, and never interpret a
delta change without checking whether theta moved the same way.

**2. Beta is measured on the edge of the filter.** The bandpass is 0.1-30 Hz, and beta is
13-30 Hz. At 30 Hz the filter has already cut power in half. So absolute beta is
systematically underestimated. The bias is the same at every visit, so *trends* in beta are
fine - but do not quote an absolute beta value as if it were unattenuated.

**3. This is not purely task-evoked power.** The 1 s analysis window starts 200 ms *before*
the stimulus, so band power reflects ongoing activity in a stimulus-locked window rather
than a purely post-stimulus response. Describe it as band power in a stimulus-locked
window, not as the response to the stimulus.

**4. Alpha and theta are the well-resolved bands.** Alpha gets 10 bins, theta 8, both
comfortably wider than the smoothing. If you need one headline result, it should come from
alpha.

**5. Absolute vs relative.** Covered in Step 5 - report both, and be suspicious of anything
that appears only in absolute power.

**6. About the Friedman test.** You mentioned it as part of the pipeline. It is worth
knowing that **there is no Friedman test anywhere in your colleague's notebook** - her
reports use a Hotelling T-squared test across the four bands to classify responders, plus a
linear regression of the acute effect on visit number. If you were told Friedman was used,
it came from a different script. Step 10 above is a genuine Friedman test, added here.

## Step 12 - Collect everything

Bundles the CSVs and figures into one zip you can download or drop into a shared drive.

In [ ]:
import zipfile, glob

zip_path = os.path.join(OUT_FOLDER, "spectral_trends_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(os.path.join(OUT_FOLDER, "*.csv")) + glob.glob(os.path.join(FIG_DIR, "*.png")):
        z.write(f, os.path.relpath(f, OUT_FOLDER))

print("wrote", zip_path)
print(f"{len(zipfile.ZipFile(zip_path).namelist())} files bundled")